In [ ]:
!git clone https://github.com/MAntonioST/assistente-medico-llm.git
%cd assistente-medico-llm
!pip uninstall -y torchaudio
!pip install -r requirements.txt
!pip install transformers==4.40.0 peft==0.10.0 accelerate bitsandbytes langchain-huggingface

In [ ]:
import logging
import sys
from google.colab import drive
drive.mount('/content/drive')

# Configuração do Logging para aparecer limpo no Colab
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - [%(name)s] - %(message)s',
    handlers=[
        logging.FileHandler("auditoria_assistente.log"),
        logging.StreamHandler(sys.stdout)
    ],
    force=True
)
logger = logging.getLogger("Setup_Modelo")
logger.info("Sistema de auditoria inicializado com sucesso.")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

caminho_adaptador = "/content/drive/MyDrive/TechChallenge3/modelo_treinado_final"
modelo_base_id = "HuggingFaceH4/zephyr-7b-beta"

logger.info("Carregando modelo base e adaptador (isso pode levar alguns minutos)...")
tokenizer = AutoTokenizer.from_pretrained(modelo_base_id)
modelo_base = AutoModelForCausalLM.from_pretrained(
    modelo_base_id,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)
modelo_treinado = PeftModel.from_pretrained(modelo_base, caminho_adaptador)

pipe = pipeline("text-generation", model=modelo_treinado, tokenizer=tokenizer, max_new_tokens=256, temperature=0.1)
llm_local = HuggingFacePipeline(pipeline=pipe)
logger.info("Modelo carregado com sucesso!")

In [ ]:
from src.agent.workflow import AssistenteMedicoApp
from src.agent.exam_agent import ExamAgent
from src.agent.protocol_agent import ProtocolAgent
from src.guardrails.medical_guardrail import MedicalGuardrail

logger = logging.getLogger("LangGraph_Orquestrador")
logger.info("Inicializando agentes...")
agente_exames = ExamAgent()
agente_protocolos = ProtocolAgent(llm=llm_local)
guardrail = MedicalGuardrail()

logger.info("Montando o LangGraph...")
app = AssistenteMedicoApp(agente_exames, agente_protocolos, guardrail)
grafo = app.get_graph()
logger.info("Sistema Multiagentes pronto para uso!")

In [ ]:
from IPython.display import Image, display

logger = logging.getLogger("Arquitetura")
logger.info("Gerando visualização do LangGraph...")
display(Image(grafo.draw_mermaid_png()))

In [ ]:
logger = logging.getLogger("Teste_Protocolos")
pergunta_protocolo = "Qual é o tratamento padrão para hipertensão?"
logger.info(f"Nova requisição recebida: {pergunta_protocolo}")

resposta = app.processar(pergunta_protocolo)
logger.info("Processamento concluído.")

print(f"\n👨‍⚕️ Usuário: {pergunta_protocolo}")
print(f"🤖 Assistente: {resposta}")

In [ ]:
logger = logging.getLogger("Teste_Exames")
pergunta_exame = "Quais são os resultados dos exames de sangue do Robert Johnson?"
logger.info(f"Nova requisição recebida: {pergunta_exame}")

resposta = app.processar(pergunta_exame)
logger.info("Processamento concluído.")

print(f"\n👨‍⚕️ Usuário: {pergunta_exame}")
print(f"🤖 Assistente: {resposta}")

In [ ]:
logger = logging.getLogger("Teste_Historico")
pergunta_historico = "Qual é o histórico clínico do paciente Robert Johnson?"
logger.info(f"Nova requisição recebida: {pergunta_historico}")

resposta = app.processar(pergunta_historico)
logger.info("Processamento concluído.")

print(f"\n👨‍⚕️ Usuário: {pergunta_historico}")
print(f"🤖 Assistente: {resposta}")